# CH 담당: 에이전트 1 거시경제 · 에이전트 3 투자심리·수급 (기초기획서 R4)

코드는 `ch_agents/` 패키지에 있고, 이 노트북은 실행·확인 화면입니다. 테스트는 `pytest members/CH/tests`로 돌립니다.

| 단계 | 하는 일 |
| --- | --- |
| 1. 데이터 | 관점마다 SQL 한 번으로 묶음을 받아 파이썬이 계산 → 숫자 장부(`M1`, `F3`, `S5`…) |
| 2. 판정 | 코드 규칙이 국면·경고·엇갈림을 판정 (숫자는 키로만) |
| 3. LLM | 장부 키만 써서 JSON 작성 → 코드 검사 → 문제 있으면 1회 수정 요청 |
| 4. 정리 | 그래도 남는 검증 안 된 문장은 빼고, LLM 실패 시 규칙 기반 대체 요약 |
| 5. Verifier | 팀 공용 fact_source_verifier로 최종 문장 수치를 장부와 1:1 대조, 주장마다 출처 태그 |

**관점별 고도화 내용** (번호는 교재 8~9장 기준. 재무 코드 `ch_agents/financial.py`는 기능을 넘겨받는 팀원이 참고용으로 씀, 7번에 임시 등록)

| 관점 | 내용 |
| --- | --- |
| 에이전트 1 거시경제 | 금리·환율·물가 국면 판정, 반도체 수출 추세, 주가와 환율·코스피·미국 금리·SOX 동조성(60·250일), 업종별 전달 경로와 업종 전용 변수(구리·철광석·LNG·주택가격·중국 수출), 규칙 기반 노출 점검표, 미확인 전제 목록(헤지·결제 시점·판가 전가력 등), 관심 변수 입력 |
| 에이전트 3 투자심리·수급 | 수급 강도(거래대금 대비)·연속 일수, 외국인 지분율 1년 위치, 문서 감성 직접 집계, 검색 관심도, VIX, 수급·가격 엇갈림, 이상 징후(집중 매매·논조 급변·관심 급증·대차 급증), 거래대금 급증, 관찰 기간 입력, 심리·수급 요약표, 규칙·사전 기준선 비교, 중복 신호(뉴스+거래량) 묶음 |

**준비 (Windows cmd)**

```
cd OpenResearch
pip install -r requirements.txt
```

`.env`에 `DATABASE_URL`, `UPSTAGE_API_KEY`가 있어야 합니다. 노트북은 `members/CH` 폴더에서 엽니다.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import Markdown, display
import pandas as pd

from ch_agents import fetch_agent_data, review_report, run_agent, run_my_agents
from ch_agents.agents import MY_AGENTS, today_kst
from ch_agents.evaluate import cross_check, data_scorecard, kospi200_tickers, llm_scorecard

TICKER = "005930"      # 분석 종목
MODE = "expert"        # expert | guide
AS_OF = today_kst()    # 기준일. 과거 시점 재현은 "2026-10-26"처럼 지정

## 1. 데이터만 확인 (LLM 호출 없음)

Solar를 부르기 전에 장부·판정·무자료를 확인합니다. 숫자가 이상하면 여기서 멈추고 SQL·계산을 고칩니다.

In [ ]:
DATA = {a: fetch_agent_data(a, TICKER, AS_OF) for a in MY_AGENTS}
for a, d in DATA.items():
    f = d.facts
    md = [f"### 에이전트 {a} · {d.period}", "", f.table_md(), "", "**코드 판정**", ""] + [f"- {x}" for x in f.flags or ["(없음)"]]
    md += ["", "**해석 가이드**", ""] + [f"- {x}" for x in f.notes or ["(없음)"]]
    md += ["", "**무자료**", ""] + [f"- {g['label']}: {g['reason']}" for g in f.gaps]
    display(Markdown("\n".join(md)))

## 1-1. DB 기반 RAG: 관점별 참고 문서 (LLM 호출 없음)

관점마다 정해 둔 용어로 `rag.document`(뉴스·블로그·카페·영문 뉴스·댓글)와 `core.disclosure`(공시 제목)를 검색합니다.
점수 = 용어 가중치 × 위치(제목 2, 요약 1) × 출처 가중치 × 최신성. 종목명이 없는 문서, 사진 기사, 중복 제목은 걸러냅니다.

- 에이전트는 이 문서를 `D번호`로만 인용합니다. 문서 속 숫자는 DB 확정값이 아니므로 출력에 쓰면 코드 검사가 지웁니다.
- 블로그·카페·댓글은 '개인 의견'으로 표시되고, 이것만으로는 '사실' 주장을 할 수 없습니다.
- 팀원 관점(2 산업 · 4 기술적 · 5 경영진·대주주 · 7 재무) 프리셋도 들어 있어 팀원이 `retrieve(관점번호, 종목코드)`로 그대로 쓸 수 있습니다.

In [ ]:
from ch_agents.rag import docs_for_prompt

for a in MY_AGENTS:
    print(f"=== 에이전트 {a} ({len(DATA[a].docs)}건) ===")
    print(docs_for_prompt(DATA[a].docs), "\n")

## 2. 교차 검증

파이썬 계산값 몇 개를 SQL로 따로 다시 계산해 비교합니다. '불일치'가 나오면 계산 로직을 의심합니다.

In [ ]:
pd.DataFrame(cross_check(TICKER, AS_OF))

## 3. 에이전트 실행

위에서 받은 `DATA`를 그대로 넘겨 DB를 다시 조회하지 않습니다. 상태는 다음 중 하나입니다.

- `완료`: 검사 통과
- `완료 (수정 1회)`: 처음 출력에 문제가 있어 한 번 고쳐 통과
- `완료 (일부 문장 제외)`: 고친 뒤에도 남은 검증 안 된 문장을 뺌
- `규칙 기반 대체`: LLM 실패 또는 남은 주장이 없어 코드 판정으로 요약

In [ ]:
res = run_agent(1, data=DATA[1], mode=MODE)
print(res["status"], f"{res['seconds']}초")
display(Markdown(res["analysis"]))

In [ ]:
res = run_agent(3, data=DATA[3], mode=MODE)
print(res["status"], f"{res['seconds']}초")
display(Markdown(res["analysis"]))

## 3-1. 데모 시나리오 (기초기획서 R4)

기획서의 입력·출력 그대로 실행합니다.

| 시나리오 | 입력 | 확인할 출력 |
| --- | --- | --- |
| ① 전문가 · 에이전트 1 | 관심 거시 변수 `focus=["환율", "금리"]` | 노출 점검표(규칙 기반 등급), 미확인 전제 목록, 반대 근거, 주장마다 `[출처: ...]`, Verifier 대조 결과 |
| ② 가이드 · 에이전트 3 | 관찰 기간 `period=10` (거래일) | 이상 징후, 기준선 비교, 중복 신호 주의, 심리·수급 요약표, 지표 해석 설명, 스스로 확인할 질문 |
| ③ 오케스트레이터 묶음 | `run_my_agents(TICKER, focus=..., period=...)` | 에이전트 1·3 출력 dict 두 개 (각각 `contract`에 데이터 계약 JSON) |

- 노출 등급과 이상 징후는 코드가 정합니다. LLM은 그 판정을 해석만 합니다.
- Verifier는 팀 공용 `agent/verifier/fact_source_verifier.py`로 최종 문장의 숫자를 장부와 1:1 대조하고, 맞지 않는 문장은 뺍니다.
- 모드 문구는 `ch_agents/prompts/expert.md`, `guide.md`에서 고칩니다.

In [ ]:
# ① 전문가 · 에이전트 1 · 관심 변수
res1 = run_agent(1, TICKER, mode="expert", as_of=AS_OF, focus=["환율", "금리"])
print(res1["status"], f"{res1['seconds']}초")
display(Markdown(res1["analysis"]))
pd.DataFrame(res1["verification"])[["문장", "ok", "unmatched", "sources"]]

In [ ]:
# ② 가이드 · 에이전트 3 · 관찰 기간 10거래일
res3 = run_agent(3, TICKER, mode="guide", as_of=AS_OF, period=10)
print(res3["status"], f"{res3['seconds']}초", "이상 징후:", len(res3["anomalies"] or []))
display(Markdown(res3["analysis"]))

In [ ]:
# ③ 오케스트레이터에 넘길 묶음 (에이전트 1·3)
# 지표 정의는 claim_stats 한 곳에서 (주장·반대근거 수, claimId 가 있는 대조 통과 수, 뺀·자른 문장 수)
outs = run_my_agents(TICKER, mode=MODE, as_of=AS_OF, focus=["환율"], period=20)
pd.DataFrame([{"관점": f"에이전트 {o['agent_id']}", "상태": o["status"], **o["claim_stats"],
               "품질 주의": len(o["quality"] or [])} for o in outs])

## 3-2. 데이터 계약 (기초기획서 R4 3장)

`res["contract"]`가 팀 공통 JSON입니다. `reportId`, 주장 `claimId`(C01…), 근거 `evidenceId`(E01…), 사실·해석·예측(`type`),
반대 근거(`role="counter"`)·하방 위험(`downsideRisks`), 미확인 상태(`unverified`), 자료 기준일(`dataDate`)·공개시각(`publishedAt`),
출처 태그(`sourceTag`)와 Verifier 결과가 들어 있습니다. 스키마: `ch_agents/schema/agent_output.schema.json`, 설명: `docs/DATA_CONTRACT.md`.

In [ ]:
import json
from ch_agents import contract

for o in outs:
    c = o["contract"]
    print(c["reportId"], "스키마 위반:", contract.validate(c) or "없음", "/ 연결 오류:", contract.link_errors(c) or "없음")

c = outs[0]["contract"]
pd.DataFrame([{"claimId": x["claimId"], "역할": x["role"], "구분": x["typeLabel"], "문장": x["text"],
               "근거 ID": ", ".join(x["evidenceIds"]), "Verifier": x["verification"]["status"]} for x in c["claims"]])

In [ ]:
# 근거(evidence)와 미확인 상태
display(pd.DataFrame(c["evidence"])[["evidenceId", "ledgerKey", "label", "display", "dataDate", "publishedAt", "sourceTag", "cited"]])
pd.DataFrame(c["unverified"])
# 파일로 저장하려면: open("contract.json", "w", encoding="utf-8").write(json.dumps(c, ensure_ascii=False, indent=2))

## 3-3. 프로필별 화면 (숙련도 L1~L3 × 성향 방어/공격)

같은 계약을 프로필별로 보여 줍니다. LLM 을 다시 부르지 않고, 수치·근거·판정은 6개 프로필에서 같습니다 (`content_set` 비교).

In [ ]:
from ch_agents import render_all
from ch_agents.renderer import content_set

PROFILE = "L1-defensive"   # L1|L2|L3 - defensive|aggressive
VIEWS = render_all(res["contract"])
print("내용 동일:", len({frozenset(content_set(v)) for v in VIEWS.values()}) == 1)
display(Markdown(VIEWS[PROFILE]["markdown"]))

## 4. 오케스트레이터에 넘길 묶음

키 구성: `agent_id`, `agent_name`, `status`, `period`, `as_of`, `result`(핵심진단·주장·긍정·리스크·미확인·다음점검·점수),
`facts`, `gaps`, `flags`, `checks`, `removed`, `attempts`, `seconds`, `analysis`(마크다운), `source`.
오케스트레이터 담당 팀원에게 이 형식을 공유하면 됩니다.

In [ ]:
MY_OUTPUTS = [run_agent(a, data=DATA[a], mode=MODE) for a in MY_AGENTS]
pd.DataFrame([{"관점": f"에이전트 {o['agent_id']}", "상태": o["status"], **o["claim_stats"],
               "숫자": len(o["facts"]), "무자료": len(o["gaps"]), "판정": len(o["flags"]), "초": o["seconds"]}
              for o in MY_OUTPUTS])

## 5. 보고서 첨삭 모드

코드가 보고서 숫자를 장부와 대조하고(일치·불일치·대조 불가), 관점마다 빠진 논점과 수정 제안을 냅니다.

In [ ]:
report_text = """
(여기에 첨삭할 보고서를 붙여 넣으세요)
"""
# report_text = open("my_report.md", encoding="utf-8").read()

review = review_report(report_text, TICKER, mode=MODE, datas=[DATA[a] for a in MY_AGENTS])
display(Markdown("### 수치 대조\n\n" + review["number_check_md"]))
for r in review["reviews"]:
    display(Markdown(f"### 에이전트 {r['agent_id']} 첨삭\n\n" + r["analysis"]))

## 6. 여러 종목 일괄 점검

데이터 단계(LLM 없음)는 KOSPI 200 전체도 돌릴 수 있습니다. LLM 단계는 비용이 들어 몇 종목만 돌립니다.
결과는 `eval_results.json`에 저장됩니다.

In [ ]:
score = pd.DataFrame(data_scorecard(kospi200_tickers(30), as_of=AS_OF))
score.pivot_table(index=["ticker", "name"], columns="agent", values=["facts", "gaps"], aggfunc="first")

In [ ]:
llm_scorecard(["005930", "000660", "035420"], mode=MODE)

## 6-1. 스냅샷 저장·재현

실제 DB 묶음을 파일로 저장하면 DB 없이 같은 결과를 다시 만들 수 있습니다. 회귀 비교나 DB 접속이 어려운 데모에 씁니다.
스냅샷 파일(`snapshot_*.json`)은 커밋하지 않습니다.


In [ ]:
from ch_agents.evaluate import capture_snapshot, load_snapshot

path = capture_snapshot(TICKER, as_of=AS_OF)
SNAP = load_snapshot(path)          # {1: AgentData, 3: AgentData}
res = run_agent(1, data=SNAP[1], mode=MODE)
print(path, res["status"])


## 7. 정답값 대조표 (삼성전자, `AS_OF = "2026-10-26"`)

2026-10-26 저녁에 같은 정의로 SQL을 따로 돌려 구한 값입니다. `AS_OF = "2026-10-26"`으로 1번 셀을 돌린 결과와 비교하세요.
그 뒤 수집이 더 되면(예: 10월 26일 수급 추가) 값이 조금 달라질 수 있습니다.

| 항목 | 정답값 |
| --- | --- |
| 주가-코스피 상관(250일) | +0.94 |
| 주가-코스피 베타(250일) | +1.30 |
| 주가-원/달러 상관(250일) | -0.01 |
| 외국인 20일 순매수 수량 | -21,805,837주 |
| 외국인 20일 순매수 강도(거래대금 대비) | -5.89% |
| 외국인 연속 순매도 일수 | 2일 |
| 외국인 지분율 20일 변화 | -0.44%p |
| 뉴스 감성(최근 7일) | -0.007 (문서 142건, 10/26 밤 추가 수집분 포함) |
| 뉴스 감성 변화(최근 7일 − 직전 8~30일) | -0.187 → '뉴스 논조 악화' 판정 |
| 검색 관심도 배율(최근 7일 ÷ 직전 90일) | 1.57배 |
| 외국인 순매수일 평균 가격(60일, 근사) | 260,349원 (순매수 22일) |
| 현재가의 외국인 순매수 평균 가격 대비 | -17.23% (종가 215,500원) |